In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class Tensor:

    def __init__(self, data, _children=(), _op=""):
        self.data = np.array(data, dtype=np.float32)
        self.grad = np.zeros_like(self.data)
        self.T = self.data.T

        self._prev = set(_children)
        self._backward = lambda: None
        self._op = _op

    def backward(self):
        # Build topological ordering of graph
        topo = []
        visited = set()

        def build(v):
            if v not in visited:
                visited.add(v)

                for child in v._prev:
                    build(child)

                topo.append(v)

        build(self)

        # d(output)/d(output) = 1
        self.grad = np.ones_like(self.data)

        # Reverse topological order
        for v in reversed(topo):
            v._backward()

    def __repr__(self) -> str:
        return f"Tensor(data={self.data}, grad={self.grad})"

    def sum_to_shape(self, grad, shape):
        """Sum gradient axes introduced by NumPy broadcasting."""

        # Remove leading dimensions added during broadcasting
        while grad.ndim > len(shape):
            grad = grad.sum(axis=0)

        # Sum dimensions where the original shape was 1
        for axis, size in enumerate(shape):
            if size == 1 and grad.shape[axis] != 1:
                grad = grad.sum(axis=axis, keepdims=True)

        return grad

    def __add__(self, other):

        other = other if isinstance(other, Tensor) else Tensor(other)

        result = Tensor(
            self.data + other.data,
            (self, other),
            "+"
        )

        """def _backward():
            self.grad += result.grad
            other.grad += result.grad"""

        def _backward():
            self.grad += self.sum_to_shape(
                result.grad, self.data.shape
            )
            other.grad += self.sum_to_shape(
                result.grad, other.data.shape
            )

        result._backward = _backward
        return result

    def __pow__(self, exponent):
        result = Tensor(
            self.data ** exponent,
            (self,),
            f"**{exponent}"
        )

        def _backward():
            self.grad += (
                exponent
                * self.data ** (exponent - 1)
                * result.grad
            )

        result._backward = _backward
        return result    

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        result = Tensor(
            self.data * other.data,
            (self, other),
            "*"
        )

        def _backward():
            self.grad += result.grad * other.data
            other.grad += result.grad * self.data

        result._backward = _backward
        return result

    __rmul__ = __mul__

    def __matmul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        result = Tensor(
            self.data @ other.data,
            (self, other),
            "@"
        )

        def _backward():
            *dims, n_features = result.grad.shape
            grad_reshaped = result.grad.reshape(-1, n_features)

            *data_dims, data_features = self.data.shape
            data_reshaped = self.data.reshape(-1, data_features)

            *other_dims, other_features = other.data.shape
            other_reshaped = other.data.reshape(-1, other_features)

            other.grad += np.matmul(data_reshaped.T, grad_reshaped)
            self.grad += np.matmul(grad_reshaped, other_reshaped.T)

        result._backward = _backward
        return result
    
    def zero_grad(self):
        self.grad = np.zeros_like(self.data)

    def sum(self):
        result = Tensor(self.data.sum(), (self,), "sum")

        def _backward():
            self.grad += np.ones_like(self.data) * result.grad

        result._backward = _backward
        return result

In [ ]:
class DenseLayer:

    def __init__(self, n_inputs, n_neurons):
        lim = np.sqrt(1 / n_inputs)
        self.weights = Tensor(
            np.random.uniform(-lim, lim, size=(n_inputs, n_neurons))
        )
        self.biases = Tensor(
            np.zeros(n_neurons)
        )

    def forward(self, inputs):
        self.inputs = inputs if isinstance(inputs, Tensor) else Tensor(inputs)
        self.output = self.inputs @ self.weights + self.biases

class ReLU:

    def forward(self, inputs):
        inputs = inputs if isinstance(inputs, Tensor) else Tensor(inputs)
        data = inputs.data

        result = Tensor(
            np.maximum(0, data),
            _children=(inputs, ),
            _op="relu"
        )

        def _backward():
            relu_grad = data.copy()
            relu_grad[data < 0] = 0
            inputs.grad += relu_grad * result.grad

        result._backward = _backward
        self.output = result

class Sigmoid:

    def forward(self, inputs):
        inputs = inputs if isinstance(inputs, Tensor) else Tensor(inputs)
        data = inputs.data

        result = Tensor(
            1 / (1 + np.exp(-1 * data)),
            _children=(inputs, ),
            _op="sigmoid"
        )

        def _backward():
            sig_out = result.data.copy()
            inputs.grad += sig_out * (1 - sig_out) * result.grad

        result._backward = _backward
        self.output = result

class BCE_Loss:

    def calculate(self, y_pred, y_true):
        y_pred = y_pred if isinstance(y_pred, Tensor) else Tensor(y_pred)
        y_true = y_true if isinstance(y_true, Tensor) else Tensor(y_true)

        predictions = y_pred.data
        targets = y_true.data

        pred_clipped = np.clip(predictions, 1e-7, 1-1e-7)
        loss = -(targets * np.log(pred_clipped) + (1 - targets) * np.log(1 - pred_clipped))

        result = Tensor(
            np.mean(np.sum(loss, axis=1)),
            _children=(y_pred,),
            _op="bce_loss"
        )

        def _backward():
            bce_grad = (pred_clipped - targets) / (pred_clipped * (1 - pred_clipped))
            y_pred.grad += bce_grad * result.grad

        # Store tensor result
        result._backward = _backward
        self.output = result

        # For display purposes
        self.value = result.data

    def backward(self):
        self.output.backward()

class MSE_Loss:

    def calculate(self, y_pred, y_true):
        y_pred = y_pred if isinstance(y_pred, Tensor) else Tensor(y_pred)
        y_true = y_true if isinstance(y_true, Tensor) else Tensor(y_true)

        predictions = y_pred.data
        targets = y_true.data

        loss = 0.5 * (predictions - targets)**2

        result = Tensor(
            np.mean(loss),
            _children=(y_pred,),
            _op="mse_loss"
        )

        def _backward():
            mse_grad = predictions - targets
            y_pred.grad += mse_grad * result.grad

        # Store tensor result
        result._backward = _backward
        self.output = result

        # For display purposes
        self.value = result.data

    def backward(self):
        self.output.backward()

In [ ]:
def plot_2d_clf_problem(X, y, h=None):
    '''
    Plots a two-dimensional labeled dataset (X,y) and, if function h(x) is given, 
    the decision surfaces.
    '''
    assert X.shape[1] == 2, "Dataset is not two-dimensional"
    if h!=None : 
        # Create a mesh to plot in
        r = 0.03  # mesh resolution
        x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
        y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
        xx, yy = np.meshgrid(np.arange(x_min, x_max, r),
                             np.arange(y_min, y_max, r))
        XX=np.c_[xx.ravel(), yy.ravel()]
        try:
            Z_test = h(XX)
            if Z_test.shape == ():
                # h returns a scalar when applied to a matrix; map explicitly
                Z = np.array(list(map(h,XX)))
            else :
                Z = Z_test
        except ValueError:
            # can't apply to a matrix; map explicitly
            Z = np.array(list(map(h,XX)))
        # Put the result into a color plot
        Z = Z.reshape(xx.shape)
        plt.contourf(xx, yy, Z, cmap=plt.cm.Pastel1)

    # Plot the dataset
    plt.scatter(X[:,0],X[:,1], c=y, cmap=plt.cm.tab20b, marker='o', s=50);

# Simple XOR dataset for classification

In [ ]:
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

y = np.array([0, 1, 1, 0]).astype(np.int64)

plot_2d_clf_problem(X, y)

In [ ]:
den1 = DenseLayer(2, 8)
sig1 = Sigmoid()
den2 = DenseLayer(8, 1)
sig2 = Sigmoid()

loss = BCE_Loss()

den1.forward(X)
sig1.forward(den1.output)
den2.forward(sig1.output)
sig2.forward(den2.output)
loss.calculate(sig2.output, y.reshape(-1, 1))

In [ ]:
epochs = 5000
lr = 5e-1

den1 = DenseLayer(2, 8)
sig1 = Sigmoid()
den2 = DenseLayer(8, 1)
sig2 = Sigmoid()
loss_fn = BCE_Loss()

for e in range(epochs):

    den1.weights.zero_grad()
    den2.weights.zero_grad()

    den1.forward(X)
    sig1.forward(den1.output)
    den2.forward(sig1.output)
    sig2.forward(den2.output)
    
    loss_fn.calculate(sig2.output, y.reshape(-1, 1))

    if not e % 1000:
        print(f'Loss: {loss_fn.value}')

    loss_fn.backward()

    den1.weights.data += -lr * den1.weights.grad
    den2.weights.data += -lr * den2.weights.grad

In [ ]:
def predict(X):
    den1.forward(X)
    sig1.forward(den1.output)
    den2.forward(sig1.output)
    sig2.forward(den2.output)
    return sig2.output.data

print(f'Accuracy: {np.mean(y.reshape(-1, 1) == np.round(predict(X)))}')

In [ ]:
plot_2d_clf_problem(X, y, lambda x: predict(x) > 0.5)

In [ ]:
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=200, factor=0.3, noise=0.1)
plot_2d_clf_problem(X, y)

In [ ]:
epochs = 1000
lr = 5e-3

den1 = DenseLayer(2, 32)
relu1 = ReLU()
den2 = DenseLayer(32, 64)
relu2 = ReLU()
den3 = DenseLayer(64, 1)
sig = Sigmoid()

layers = [den1, den2, den3]

loss_fn = BCE_Loss()

for e in range(epochs):

    den1.forward(X)
    relu1.forward(den1.output)
    den2.forward(relu1.output)
    relu2.forward(den2.output)
    den3.forward(relu2.output)
    sig.forward(den3.output)
    
    loss_fn.calculate(sig.output, y.reshape(-1, 1))

    if not e % 100:
        print(f'Loss: {loss_fn.value}')

    loss_fn.backward()

    # Update weights and biases
    for layer in layers:
        layer.weights.data -= lr * layer.weights.grad
        layer.biases.data -= lr * layer.biases.grad

    # Reset parameter gradients
    for layer in layers:
        layer.weights.zero_grad()
        layer.biases.zero_grad()

In [ ]:
def predict(X):
    den1.forward(X)
    relu1.forward(den1.output)
    den2.forward(relu1.output)
    relu2.forward(den2.output)
    den3.forward(relu2.output)
    sig.forward(den3.output)
    return sig.output.data

print(f'Accuracy: {np.mean(y.reshape(-1, 1) == np.round(predict(X)))}')

In [ ]:
plot_2d_clf_problem(X, y, lambda x: predict(x) > 0.5)

In [ ]:
from sklearn.datasets import make_regression

X, y = make_regression(n_features=1, n_informative=1, n_targets=1, noise=5.3, random_state=52)

plt.scatter(X, y)
plt.show()

In [ ]:
epochs = 1000
lr = 1e-3

den1 = DenseLayer(1, 1)
layers = [den1]
loss_fn = MSE_Loss()

for e in range(epochs):

    den1.forward(X)
    
    loss_fn.calculate(den1.output, y.reshape(-1, 1))

    if not e % 100:
        print(f'Loss: {loss_fn.value}')

    loss_fn.backward()

    # Update weights and biases
    for layer in layers:
        layer.weights.data -= lr * layer.weights.grad
        layer.biases.data -= lr * layer.biases.grad

    # Reset parameter gradients
    for layer in layers:
        layer.weights.zero_grad()
        layer.biases.zero_grad()

In [ ]:
def predict(X):
    den1.forward(X)
    return den1.output.data

In [ ]:
plt.scatter(X, y)
plt.plot(X, predict(X), color='red')
plt.show()